# B2-023 — Practice p10

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** denoising-diffusion-probabilistic-models

## Task

Implement the ε-prediction training loss with an explicit mean reduction.

**Contract.** Include your p09 `make_schedule` and `q_sample` (same contract).
`ddpm_loss(model, x0, t, eps, alpha_bar)`: with `T = alpha_bar.shape[0] - 1`, compute `x_t = q_sample(x0, t, eps, alpha_bar)` by calling the module-level `q_sample` (the clean rows enter **only** through its `x0` argument), then `t_scaled = (t.to(x0.dtype) / T).unsqueeze(1)` of shape `(B,1)`, then `eps_hat = model(x_t, t_scaled)`, and return the 0-dimensional tensor `((eps_hat - eps) ** 2).mean()`, the mean over all $B\times d$ entries. Raise `ValueError` if `eps_hat.shape != eps.shape`.

**Banned:** `F.mse_loss`, `nn.MSELoss`, any `reduction=` argument, `.sum()` in the loss, and drawing random numbers inside `ddpm_loss`.

**Probes** (literals below, `schedule = make_schedule(5, 0.1, 0.5)`, `ATOL = RTOL = 1e-6`):

1. With the supplied `ToyEps` model, `ddpm_loss(ToyEps(), X0, T_IDX, EPS, schedule["alpha_bar"])` equals `1.1080242395`.
2. With the supplied `ZeroEps` model, the loss equals `EPS.pow(2).mean()` $=5.45/6=0.9083333$.
3. **Spy.** A recording model shows that the network received `t_scaled` of shape `(3,1)`, dtype float32, values exactly `[[0.2], [0.6], [1.0]]`, and an `x_t` that is `torch.equal` to `q_sample(X0, T_IDX, EPS, schedule["alpha_bar"])`.
4. Two local wrong variants each differ from probe 1 by more than `0.1`: (a) the regression target `x0` instead of `eps` (about `1.4600834846`); (b) a sum instead of a mean over entries.
5. With `torch.manual_seed(SEED)` immediately before `model = Denoiser()` (the p18 network: `class Denoiser(nn.Module)` creates exactly `self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))`; `forward(x_t, t_scaled)` returns `self.net(torch.cat([x_t, t_scaled], dim=1))` for `x_t` of shape `(B,2)` and `t_scaled` of shape `(B,1)`.), `ddpm_loss(model, X0, T_IDX, EPS, schedule["alpha_bar"]).backward()` gives a finite gradient for every parameter, at least one of them nonzero, and the loss has shape `()`.
6. A model whose output has shape `(3,1)` raises `ValueError`.

Explain in two sentences why the reduction must be stated explicitly in a training contract.

**Required answer-check assertions.** Probes 1–6.

In [ ]:
import torch
from torch import nn

SEED = 20261015
X0 = torch.tensor([[1.0, -2.0], [0.5, 0.5], [-1.0, 3.0]])
T_IDX = torch.tensor([1, 3, 5])
EPS = torch.tensor([[0.5, 0.0], [-1.0, 2.0], [0.2, -0.4]])


class ToyEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return 0.5 * x_t + t_scaled


class ZeroEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return torch.zeros_like(x_t)


ATOL = 1e-6
RTOL = 1e-6

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.